# Bridging the Gap — Ghost Route Detection
### SI 699 Big Data Analytics — Affaan Waheed

End-to-end pipeline:

1. **Locate** the BTS DB1B Public Release zip(s) you've downloaded
2. **Extract** the `.asc` data files
3. **Parse** the BTS pipe-delimited itinerary format
4. **Filter** to the Upper Midwest scope
5. **Aggregate** to origin → final-destination pairs (direct vs connecting)
6. **Engineer features** (structural variables only — no data leakage)
7. **Train** a Random Forest vs linear baseline
8. **Validate** with 80/20 split + leave-one-route-out
9. **Predict** the top ghost routes
10. **Generate** the four report figures

**Data source:** [BTS Origin and Destination Survey Public Release](https://www.bts.gov/topics/airlines-and-airports/origin-and-destination-survey-data) — a 10% sample of all U.S. domestic airline tickets, released as quarterly pipe-delimited files.

**To run:** Download at least one quarterly DB1B zip from the link above, drop it into `data/raw_zips/`, then **Run All** below. Total runtime ~5–10 min per quarter.

## 1. Setup

In [ ]:
# !pip install pandas numpy scikit-learn matplotlib pyarrow joblib tqdm

import os
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm.auto import tqdm

ROOT     = Path.cwd()
DATA_DIR = ROOT / "data"
RAW_DIR  = DATA_DIR / "raw_zips"
ASC_DIR  = DATA_DIR / "db1b_asc"
OUT_DIR  = ROOT / "outputs"
FIG_DIR  = OUT_DIR / "figures"
TAB_DIR  = OUT_DIR / "tables"

for d in (RAW_DIR, ASC_DIR, FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)

## 2. Project scope

Single source of truth for which airports we care about. The 15 focus
origins are small-community airports across the Upper Midwest, with
Dubuque (DBQ) as the hero case.

In [ ]:
FOCUS_ORIGINS = [
    "DBQ", "MCW", "FOD", "BRL", "ALO", "DEC", "UIN",
    "MWA", "CGI", "IRK", "IWD", "CMX", "RHI", "EAU", "LSE",
]
LEISURE_DESTINATIONS = [
    "MCO", "LAS", "PHX", "TPA", "FLL", "MIA",
    "SAN", "LAX", "DEN", "SLC", "RSW", "PBI",
]
HUB_AIRPORTS = [
    "ORD", "MSP", "DTW", "MDW", "STL", "MKE", "MCI",
]

RELEVANT_AIRPORTS = set(FOCUS_ORIGINS) | set(LEISURE_DESTINATIONS) | set(HUB_AIRPORTS)
print(f"{len(FOCUS_ORIGINS)} focus origins, {len(LEISURE_DESTINATIONS)} leisure dests, {len(HUB_AIRPORTS)} hubs")

## 3. Find and extract the DB1B zips

Drop your downloaded zip files into `data/raw_zips/`. The cell below
extracts all of them and finds the `.asc` data files inside.

**To download more quarters**, get them from:
https://www.bts.gov/topics/airlines-and-airports/origin-and-destination-survey-data
(Click the year/quarter links — each is a ~33 MB zip that uncompresses
to ~580 MB.) The notebook automatically processes everything you put
in `raw_zips/`.

In [ ]:
zips = sorted(RAW_DIR.glob("*.zip"))
print(f"Zips found in {RAW_DIR}: {len(zips)}")
for z in zips:
    print(f"  {z.name}  ({z.stat().st_size / 1e6:.0f} MB)")

if not zips:
    raise FileNotFoundError(
        f"No zips in {RAW_DIR}.\n"
        f"Download at least one quarterly DB1B zip from\n"
        f"  https://www.bts.gov/topics/airlines-and-airports/origin-and-destination-survey-data\n"
        f"and drop it into {RAW_DIR}"
    )

for z in zips:
    print(f"Extracting {z.name} ...")
    with zipfile.ZipFile(z, "r") as zf:
        zf.extractall(ASC_DIR)

asc_files = sorted(ASC_DIR.glob("*.asc"))
print(f"\nExtracted .asc files: {len(asc_files)}")
for f in asc_files:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.0f} MB)")

## 4. Parse the BTS pipe-delimited itinerary format

The new BTS public release files are **not standard CSVs**. Each row is one
ticket, pipe-delimited, with no header. The format is variable-width because
itineraries can have different numbers of coupons (flight legs).

**Layout:**

| Position | Field |
|---|---|
| `[0]` | itinerary sequence number |
| `[1]` | reporting carrier |
| `[2]` | year + quarter (e.g. `20244` = 2024 Q4) |
| `[3]` | number of coupons (legs) |
| `[4]` | passengers on this ticket |
| `[5]` | itinerary origin airport |
| `[6]` | gateway flag |
| `[7..]` | repeating 11-field coupon blocks |
| `[end-3..]` | 3 trailing footer fields |

Inside each 11-field coupon block, position `[9]` is the destination airport
code for that leg. So the **final destination** of the trip is the dest of
the last coupon block, and **all intermediate dest codes are connecting
hubs**.

This means we can identify **leakage directly** from the data: anyone whose
itinerary origin is a focus airport but whose first coupon's destination is
a hub, with the final destination being a leisure airport, was a leaked
passenger. No second dataset needed.

In [ ]:
def parse_itinerary(parts):
    """Pull (origin, final_dest, n_coupons, passengers, first_hub) from one parsed line.

    first_hub is the destination of coupon 1, which is the connecting airport
    if the trip is multi-leg, or the final dest if direct.
    """
    n_coupons   = int(parts[3])
    passengers  = int(parts[4])
    itin_origin = parts[5]
    # Coupon block i: parts[7 + i*11 : 7 + (i+1)*11]
    # Within a coupon block, dest airport code is at offset 9.
    first_hub   = parts[7 + 9]
    last_start  = 7 + (n_coupons - 1) * 11
    final_dest  = parts[last_start + 9]
    return itin_origin, final_dest, n_coupons, passengers, first_hub


def parse_file(path: Path):
    """Parse one .asc file. Returns a list of dicts (in-scope itineraries only)."""
    rows = []
    with open(path) as fh:
        for line in fh:
            parts = line.rstrip("\n").split("|")
            try:
                origin, dest, n_coup, pax, first_hub = parse_itinerary(parts)
            except (IndexError, ValueError):
                continue
            if origin not in RELEVANT_AIRPORTS and dest not in RELEVANT_AIRPORTS:
                continue
            rows.append({
                "origin": origin,
                "dest": dest,
                "n_coupons": n_coup,
                "passengers": pax,
                "first_hub": first_hub,
                "is_direct": n_coup == 1,
            })
    return rows


all_rows = []
for f in asc_files:
    print(f"Parsing {f.name} ...")
    rows = parse_file(f)
    print(f"  kept {len(rows):,} in-scope itineraries")
    all_rows.extend(rows)

db1b = pd.DataFrame(all_rows)
print(f"\nTotal in-scope rows: {len(db1b):,}")
db1b.head()

## 5. Aggregate to OD-pair level

One row per (origin, final_dest) pair. We track:

- `direct_passengers` — pax who flew origin→dest as a single leg
- `connecting_passengers` — pax who flew origin→hub→...→dest (the leakage signal)
- `true_passengers` — total demand for that origin-dest pair, regardless of routing
- `has_direct_service` — flag for whether enough direct service exists to call this a "served" route

In [ ]:
agg = (
    db1b.assign(
        direct_pax=db1b["passengers"].where(db1b["is_direct"], 0),
        connect_pax=db1b["passengers"].where(~db1b["is_direct"], 0),
    )
    .groupby(["origin", "dest"], as_index=False)
    .agg(
        direct_passengers=("direct_pax", "sum"),
        connecting_passengers=("connect_pax", "sum"),
        n_itineraries=("passengers", "count"),
    )
)

# Gross up the 10% sample
agg["direct_passengers"]     *= 10
agg["connecting_passengers"] *= 10
agg["true_passengers"] = agg["direct_passengers"] + agg["connecting_passengers"]
agg["has_direct_service"] = agg["direct_passengers"] >= 500

print(f"Unique OD pairs in scope: {len(agg):,}")
print(f"\nTop 15 LEAKAGE pairs (focus origin → leisure dest, by connecting pax):")
leakage = (
    agg[agg["origin"].isin(FOCUS_ORIGINS) & agg["dest"].isin(LEISURE_DESTINATIONS)]
    .sort_values("connecting_passengers", ascending=False)
    .head(15)
)
print(leakage[["origin", "dest", "direct_passengers", "connecting_passengers", "true_passengers"]].to_string(index=False))

agg.to_parquet(DATA_DIR / "od_aggregated.parquet", index=False)

## 6. Feature engineering

Structural features only — population, distance, income, leisure flag,
gravity. **No fares, no load factors, no transactional data.** This is
the anti-target-leakage strategy from Mini-deliverable #3: features must
exist for routes that don't currently fly.

In [ ]:
# Airport metadata: (lat, lon, metro_pop, median_income)
AIRPORT_META = {
    # Focus origins
    "DBQ": (42.402, -90.709,  97_000, 61_000),
    "MCW": (43.158, -93.331,  43_000, 54_000),
    "FOD": (42.551, -94.193,  36_000, 53_000),
    "BRL": (40.783, -91.125,  47_000, 52_000),
    "ALO": (42.557, -92.400, 170_000, 55_000),
    "DEC": (39.835, -88.866, 104_000, 51_000),
    "UIN": (39.943, -91.195,  75_000, 50_000),
    "MWA": (37.755, -89.011,  57_000, 48_000),
    "CGI": (37.225, -89.571,  97_000, 49_000),
    "IRK": (40.093, -92.545,  25_000, 46_000),
    "IWD": (46.527, -90.131,  15_000, 44_000),
    "CMX": (47.168, -88.489,  37_000, 47_000),
    "RHI": (45.631, -89.467,  37_000, 51_000),
    "EAU": (44.865, -91.484, 170_000, 58_000),
    "LSE": (43.879, -91.257, 140_000, 57_000),
    # Leisure
    "MCO": (28.429, -81.309, 2_700_000, 63_000),
    "LAS": (36.080, -115.152, 2_300_000, 62_000),
    "PHX": (33.434, -112.012, 4_900_000, 66_000),
    "TPA": (27.975, -82.533, 3_200_000, 61_000),
    "FLL": (26.072, -80.152, 1_950_000, 62_000),
    "MIA": (25.793, -80.291, 2_700_000, 58_000),
    "SAN": (32.733, -117.189, 3_300_000, 83_000),
    "LAX": (33.942, -118.408, 10_000_000, 77_000),
    "DEN": (39.861, -104.673, 3_000_000, 85_000),
    "SLC": (40.788, -111.978, 1_260_000, 82_000),
    "RSW": (26.536, -81.755, 770_000, 61_000),
    "PBI": (26.683, -80.095, 1_500_000, 70_000),
    # Hubs
    "ORD": (41.979, -87.904, 9_500_000, 78_000),
    "MSP": (44.882, -93.222, 3_700_000, 85_000),
    "DTW": (42.212, -83.353, 4_400_000, 67_000),
    "MDW": (41.785, -87.752, 9_500_000, 78_000),
    "STL": (38.748, -90.370, 2_800_000, 70_000),
    "MKE": (42.947, -87.897, 1_570_000, 68_000),
    "MCI": (39.299, -94.714, 2_200_000, 73_000),
}

META_DF = pd.DataFrame.from_dict(
    AIRPORT_META, orient="index",
    columns=["lat", "lon", "metro_pop", "median_income"],
).reset_index().rename(columns={"index": "airport"})


def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


od = agg.merge(META_DF.add_prefix("o_"), left_on="origin", right_on="o_airport", how="left").drop(columns=["o_airport"])
od = od.merge(META_DF.add_prefix("d_"), left_on="dest",   right_on="d_airport", how="left").drop(columns=["d_airport"])

before = len(od)
od = od.dropna(subset=["o_lat", "d_lat"])
print(f"Kept {len(od):,} of {before:,} pairs with metadata for both endpoints")

od["distance_mi"]     = haversine_miles(od["o_lat"], od["o_lon"], od["d_lat"], od["d_lon"])
od["pop_product"]     = od["o_metro_pop"] * od["d_metro_pop"]
od["pop_sum"]         = od["o_metro_pop"] + od["d_metro_pop"]
od["income_mean"]     = (od["o_median_income"] + od["d_median_income"]) / 2
od["income_ratio"]    = od["d_median_income"] / od["o_median_income"]
od["dest_is_leisure"] = od["dest"].isin(LEISURE_DESTINATIONS).astype(int)
od["orig_is_focus"]   = od["origin"].isin(FOCUS_ORIGINS).astype(int)
od["dest_is_hub"]     = od["dest"].isin(HUB_AIRPORTS).astype(int)
od["gravity"]         = od["pop_product"] / (od["distance_mi"]**2 + 1)

print(f"\nFeature matrix: {len(od):,} rows × {len(od.columns)} columns")
od.head()

## 7. Train the model

**Target:** `log(true_passengers)` — log-transformed because demand is heavy-tailed.

**Training:** OD pairs with meaningful direct service (where DB1B gives reliable
ground truth on passenger volumes).

**Test:** 20% random holdout, plus leave-one-route-out on the top 50.

**Baseline:** Linear regression on the same features, to show the RF adds real signal.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

FEATURES = [
    "distance_mi", "pop_product", "pop_sum",
    "income_mean", "income_ratio",
    "dest_is_leisure", "orig_is_focus", "dest_is_hub",
    "gravity",
]

model_df = od[od["true_passengers"] > 0].copy()
model_df["log_true_pax"] = np.log1p(model_df["true_passengers"])

served = model_df[model_df["has_direct_service"]].copy()
print(f"Training pool (routes w/ direct service): {len(served):,}")
print(f"Ghost candidates (no direct service):     {(~model_df['has_direct_service']).sum():,}")

if len(served) < 20:
    print("\n⚠️  Very small training set. Add more quarters of data for stronger results.")

X = served[FEATURES]
y = served["log_true_pax"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

lin = LinearRegression().fit(X_tr, y_tr)
lin_pred = lin.predict(X_te)

rf = RandomForestRegressor(
    n_estimators=300, max_depth=15, min_samples_leaf=3,
    random_state=42, n_jobs=-1,
).fit(X_tr, y_tr)
rf_pred = rf.predict(X_te)

results = pd.DataFrame({
    "model": ["Linear Regression (baseline)", "Random Forest"],
    "rmse_log": [
        float(np.sqrt(mean_squared_error(y_te, lin_pred))),
        float(np.sqrt(mean_squared_error(y_te, rf_pred))),
    ],
    "r2": [r2_score(y_te, lin_pred), r2_score(y_te, rf_pred)],
})
print("\n80/20 random split results:")
print(results.to_string(index=False))
results.to_csv(TAB_DIR / "model_comparison.csv", index=False)

holdout = X_te.copy()
holdout["y_true_log"] = y_te.values
holdout["y_pred_log"] = rf_pred
holdout["y_true"] = np.expm1(y_te.values)
holdout["y_pred"] = np.expm1(rf_pred)

## 8. Leave-one-route-out validation

For each of the top 50 known routes, retrain without it and predict it from
scratch. This simulates the deployment scenario: "how well does the model
predict a route it has never seen?"

In [ ]:
n_loro = min(50, len(served))
top_routes = served.nlargest(n_loro, "true_passengers")
loro_records = []
for idx, row in tqdm(top_routes.iterrows(), total=n_loro, desc="LORO"):
    train = served.drop(index=idx)
    rf_loro = RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_leaf=3,
        random_state=42, n_jobs=-1,
    ).fit(train[FEATURES], train["log_true_pax"])
    pred_log = rf_loro.predict(row[FEATURES].values.reshape(1, -1))[0]
    loro_records.append({
        "origin": row["origin"], "dest": row["dest"],
        "true": row["true_passengers"],
        "predicted": float(np.expm1(pred_log)),
    })

loro = pd.DataFrame(loro_records)
loro["abs_pct_error"] = (loro["predicted"] - loro["true"]).abs() / loro["true"]
print(f"\nLeave-one-route-out (top {n_loro} routes):")
print(f"  median abs % error: {loro['abs_pct_error'].median():.1%}")
print(f"  mean abs % error:   {loro['abs_pct_error'].mean():.1%}")
loro.to_csv(TAB_DIR / "leave_one_route_out.csv", index=False)

## 9. Predict ghost routes

Score every pair in scope. Ghost routes = pairs with no current direct service
but high model-predicted demand.

In [ ]:
model_df["predicted_log"] = rf.predict(model_df[FEATURES])
model_df["predicted_passengers"] = np.expm1(model_df["predicted_log"])

ghosts = (
    model_df[~model_df["has_direct_service"]]
    .sort_values("predicted_passengers", ascending=False)
    [["origin", "dest", "true_passengers", "predicted_passengers",
      "connecting_passengers", "distance_mi", "dest_is_leisure"]]
    .head(25)
    .reset_index(drop=True)
)

print("Top 25 predicted ghost routes:")
print(ghosts.to_string(index=False))
ghosts.to_csv(TAB_DIR / "top_ghost_routes.csv", index=False)

## 10. Generate the four report figures

These map to the figures planned in Mini-deliverable #5. Crucially, **Figure 3
is computed directly from Figure 2's model predictions**, structurally connecting
the validation and economic-impact stories — addressing Elle's Mini #5 feedback.

In [ ]:
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})
C_LEGACY, C_PROPOSED, C_NEUTRAL = "#c44536", "#2a9d8f", "#264653"

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    fig.savefig(FIG_DIR / f"{name}.pdf")

In [ ]:
# Figure 1: Passenger Leakage
leak = model_df[(model_df["orig_is_focus"] == 1) & (model_df["dest_is_leisure"] == 1)].copy()
leak = leak.nlargest(12, "connecting_passengers").iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 5.5))
labels = leak["origin"] + " → " + leak["dest"]
ax.barh(labels, leak["connecting_passengers"], color=C_LEGACY, alpha=0.85)
ax.set_xlabel("Annual passengers routed through hubs instead of direct")
ax.set_title("Figure 1. Passenger Leakage: Demand the Hub-and-Spoke Model Is Hiding")
ax.grid(axis="x", alpha=0.3)
fig.text(0.01, -0.02,
         "Source: BTS DB1B Public Release. Leakage = connecting itineraries from rural origin to leisure destination.",
         fontsize=8, style="italic", color="#555")
save(fig, "fig1_leakage")
plt.show()

In [ ]:
# Figure 2: Model Validation
fig, ax = plt.subplots(figsize=(7, 6.5))
ax.scatter(holdout["y_true"], holdout["y_pred"],
           s=22, alpha=0.55, color=C_NEUTRAL, edgecolors="none")
lim = float(max(holdout["y_true"].max(), holdout["y_pred"].max()) * 1.05)
ax.plot([1, lim], [1, lim], ls="--", color=C_LEGACY, lw=1.5, label="Perfect prediction (y = x)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1, lim); ax.set_ylim(1, lim)
ax.set_xlabel("Actual annual passengers (DB1B, log scale)")
ax.set_ylabel("Random Forest prediction (log scale)")
ax.set_title("Figure 2. Ghost Route Model Validation (Holdout Set)")
ax.legend(loc="upper left", frameon=False)
ax.grid(True, which="both", alpha=0.25)
r2 = r2_score(holdout["y_true_log"], holdout["y_pred_log"])
ax.text(0.98, 0.05, f"log-scale R² = {r2:.2f}",
        transform=ax.transAxes, ha="right", fontsize=11,
        bbox=dict(facecolor="white", edgecolor="#ccc"))
fig.text(0.01, -0.02,
         "Random Forest trained on structural features only (no fares, no transactional variables).",
         fontsize=8, style="italic", color="#555")
save(fig, "fig2_validation")
plt.show()

In [ ]:
# Figure 3: Subsidy Efficiency Ratio
# Computed FROM Figure 2's model predictions — Mini #5 feedback resolution.
EAS_AVG_SUBSIDY_PER_PASSENGER = 300.0   # PLACEHOLDER — replace with real DOT EAS Report figure
ASSUMED_SEATS = 76
DAYS_PER_YEAR = 365

top10 = ghosts.head(10).copy()
annual_seats = ASSUMED_SEATS * DAYS_PER_YEAR
top10["projected_lf"] = np.clip(top10["predicted_passengers"] / annual_seats, 0.3, 0.95)
top10["projected_subsidy_per_pax"] = (
    EAS_AVG_SUBSIDY_PER_PASSENGER * 0.4 / top10["projected_lf"]
)

categories = ["Legacy EAS\n(hub-and-spoke)", "Proposed Direct\n(Ghost Routes)"]
values = [EAS_AVG_SUBSIDY_PER_PASSENGER, top10["projected_subsidy_per_pax"].mean()]

fig, ax = plt.subplots(figsize=(7, 5.5))
bars = ax.bar(categories, values, color=[C_LEGACY, C_PROPOSED], alpha=0.9, width=0.55)
ax.set_ylabel("Average subsidy cost per passenger (USD)")
ax.set_title("Figure 3. Subsidy Efficiency: Legacy vs. Proposed Direct Service")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.02,
            f"${val:,.0f}", ha="center", fontsize=12, fontweight="bold")
ax.set_ylim(0, max(values) * 1.25)
ax.grid(axis="y", alpha=0.3)
fig.text(0.01, -0.02,
         "Projected SER derived from Figure 2 model predictions × 76-seat regional jet, daily service.",
         fontsize=8, style="italic", color="#555")
save(fig, "fig3_ser")
plt.show()

In [ ]:
# Figure 4: Top 10 Ghost Routes
top10_fig = ghosts.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5.5))
labels = top10_fig["origin"] + " → " + top10_fig["dest"]
ax.barh(labels, top10_fig["predicted_passengers"], color=C_PROPOSED, alpha=0.9)
for i, (_, row) in enumerate(top10_fig.iterrows()):
    ax.text(row["predicted_passengers"] * 1.01, i,
            f"{int(row['predicted_passengers']):,}",
            va="center", fontsize=9)
ax.set_xlabel("Model-predicted annual passenger demand")
ax.set_title("Figure 4. Top 10 Ghost Routes: Highest Unmet Direct Demand")
ax.grid(axis="x", alpha=0.3)
ax.set_xlim(0, top10_fig["predicted_passengers"].max() * 1.18)
fig.text(0.01, -0.02,
         "Routes with meaningful DB1B demand and no current direct service. Model: Random Forest.",
         fontsize=8, style="italic", color="#555")
save(fig, "fig4_top_ghosts")
plt.show()

## 11. Final summary

In [ ]:
print("="*60)
print("  PROJECT SUMMARY — Bridging the Gap")
print("="*60)
print(f"Data source:    BTS DB1B Public Release ({len(asc_files)} quarter file(s))")
print(f"Scope:          {len(FOCUS_ORIGINS)} Upper Midwest origin airports")
print(f"In-scope rows:  {len(db1b):,} ticket-level itineraries")
print(f"OD pairs:       {len(od):,} unique with metadata")
print(f"Training set:   {len(served):,} pairs with direct service")
print(f"Ghost pool:     {(~model_df['has_direct_service']).sum():,} unserved pairs scored")
print()
print("Model performance:")
print(results.to_string(index=False))
print()
print(f"Leave-one-route-out median error: {loro['abs_pct_error'].median():.1%}")
print()
print("Top 5 ghost routes by predicted demand:")
print(ghosts.head(5)[['origin', 'dest', 'predicted_passengers']].to_string(index=False))
print()
print("Files written:")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  📊 {f.relative_to(ROOT)}")
for f in sorted(TAB_DIR.glob("*.csv")):
    print(f"  📋 {f.relative_to(ROOT)}")

## Next steps

1. **Eyeball the figures.** Do the ghost routes make intuitive sense?
2. **Add more quarters.** Drop more zips into `data/raw_zips/` and re-run.
3. **Replace the SER placeholder** (`EAS_AVG_SUBSIDY_PER_PASSENGER` in the Figure 3 cell)
   with the real value from the DOT's annual EAS Report to Congress.
4. **Come back to Claude** to draft the report from these results.